<a href="https://colab.research.google.com/github/BerkKilincer-dev/SoftIto/blob/main/ML/overfitting_regularizasyon_cv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aşırı Öğrenme, Regularizasyon, Veri Sızıntısı ve Çapraz Doğrulama

Bir modelin eğitim verisinde iyi olması yetmez; **hiç görmediği veride** de iyi olması gerekir. Bu not defteri genelleme ile ilgili dört temel konuyu uygulamalı olarak işler.

| # | Konu | Soru |
|---|------|------|
| 1 | Overfitting / Underfitting | Model ezber mi yapıyor, yoksa hiç mi öğrenemiyor? |
| 2 | L1 (Lasso) vs L2 (Ridge) | Ağırlıkları cezalandırarak ezberi nasıl engelleriz? |
| 3 | Veri Sızıntısı ve `StandardScaler` | Test verisinden eğitime farkında olmadan bilgi nasıl sızar? |
| 4 | K-Fold Cross-Validation | Model performansını güvenilir şekilde nasıl ölçeriz? |

> Tüm veri setleri sentetik ya da scikit-learn ile birlikte geldiği için ek kurulum veya indirme gerekmez.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression, make_classification, load_breast_cancer
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge, LogisticRegression
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import (train_test_split, KFold, StratifiedKFold,
                                     cross_val_score, GridSearchCV)
from sklearn.metrics import mean_squared_error, accuracy_score

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams["figure.dpi"] = 100

---
## 1) Overfitting ve Underfitting

| Durum | Eğitim hatası | Test hatası | Sebep | Çözüm |
|---|---|---|---|---|
| **Underfitting** (yetersiz öğrenme) | Yüksek | Yüksek | Model çok basit → **yüksek bias** | Daha karmaşık model, daha fazla özellik |
| **İyi uyum** | Düşük | Düşük | Karmaşıklık veriye uygun | — |
| **Overfitting** (aşırı öğrenme) | Çok düşük | Yüksek | Model gürültüyü ezberliyor → **yüksek varyans** | Daha fazla veri, regularizasyon, daha basit model, erken durdurma |

Bunu görmek için `sin(2πx)` fonksiyonundan gürültülü 30 nokta üretip farklı dereceli polinomlarla uyduracağız.

In [ ]:
def gercek_fonksiyon(x):
    return np.sin(2 * np.pi * x)

n_egitim = 30
x_egitim = np.sort(np.random.rand(n_egitim))
y_egitim = gercek_fonksiyon(x_egitim) + np.random.normal(0, 0.2, n_egitim)

x_test = np.sort(np.random.rand(300))
y_test = gercek_fonksiyon(x_test) + np.random.normal(0, 0.2, 300)

def polinom_model(derece):
    return make_pipeline(PolynomialFeatures(derece), LinearRegression())

In [ ]:
x_cizim = np.linspace(0, 1, 400)
fig, eksenler = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
senaryolar = [(1, "Underfitting (derece = 1)"),
              (4, "İyi uyum (derece = 4)"),
              (15, "Overfitting (derece = 15)")]

for ax, (derece, baslik) in zip(eksenler, senaryolar):
    model = polinom_model(derece).fit(x_egitim[:, None], y_egitim)
    egitim_mse = mean_squared_error(y_egitim, model.predict(x_egitim[:, None]))
    test_mse = mean_squared_error(y_test, model.predict(x_test[:, None]))

    ax.scatter(x_egitim, y_egitim, s=25, label="eğitim verisi")
    ax.plot(x_cizim, gercek_fonksiyon(x_cizim), "g--", label="gerçek fonksiyon")
    ax.plot(x_cizim, model.predict(x_cizim[:, None]), "r", lw=2, label="model")
    ax.set_ylim(-2, 2)
    ax.set_title(f"{baslik}\neğitim MSE = {egitim_mse:.3f} | test MSE = {test_mse:.3f}")
    ax.legend(fontsize=8, loc="lower left")

plt.tight_layout()
plt.show()

### Model karmaşıklığı – hata eğrisi

Derece arttıkça eğitim hatası **hep** düşer, ama test hatası önce düşer sonra yükselir. Bu U şeklindeki eğri **bias–variance dengesinin** (bias-variance tradeoff) görsel karşılığıdır.

In [ ]:
dereceler = range(1, 16)
egitim_hatalari, test_hatalari = [], []

for d in dereceler:
    model = polinom_model(d).fit(x_egitim[:, None], y_egitim)
    egitim_hatalari.append(mean_squared_error(y_egitim, model.predict(x_egitim[:, None])))
    test_hatalari.append(mean_squared_error(y_test, model.predict(x_test[:, None])))

en_iyi = dereceler[int(np.argmin(test_hatalari))]

plt.figure(figsize=(9, 4.5))
plt.semilogy(dereceler, egitim_hatalari, "o-", label="Eğitim MSE")
plt.semilogy(dereceler, test_hatalari, "s-", label="Test MSE")
plt.axvline(en_iyi, color="gray", ls=":", label=f"en iyi derece = {en_iyi}")
plt.axvspan(0.5, 2.5, alpha=0.1, color="blue")
plt.axvspan(10.5, 15.5, alpha=0.1, color="red")
plt.text(1.0, plt.ylim()[1] * 0.4, "underfitting", color="blue")
plt.text(11.5, plt.ylim()[1] * 0.4, "overfitting", color="red")
plt.xlabel("Polinom derecesi (model karmaşıklığı)")
plt.ylabel("MSE (log ölçek)")
plt.title("Bias–Variance Dengesi")
plt.legend()
plt.show()

### Daha fazla veri overfitting'i azaltır

Aynı derece-15 modeli 30 yerine 300 noktayla eğitirsek ezberleyecek kadar "boşluk" kalmaz.

In [ ]:
fig, eksenler = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, n in zip(eksenler, [30, 300]):
    x_n = np.sort(np.random.rand(n))
    y_n = gercek_fonksiyon(x_n) + np.random.normal(0, 0.2, n)
    model = polinom_model(15).fit(x_n[:, None], y_n)
    test_mse = mean_squared_error(y_test, model.predict(x_test[:, None]))
    ax.scatter(x_n, y_n, s=8, alpha=0.6)
    ax.plot(x_cizim, model.predict(x_cizim[:, None]), "r", lw=2)
    ax.plot(x_cizim, gercek_fonksiyon(x_cizim), "g--")
    ax.set_ylim(-2, 2)
    ax.set_title(f"Derece 15, n = {n} → test MSE = {test_mse:.3f}")
plt.tight_layout()
plt.show()

---
## 2) L1 (Lasso) ve L2 (Ridge) Regularizasyon

Regularizasyon, kayıp fonksiyonuna **ağırlıkların büyüklüğüne bağlı bir ceza** ekler. Model büyük ağırlıklar kullanarak gürültüyü ezberlemek yerine daha sade bir çözüme zorlanır.

$$\text{Ridge (L2)}:\quad \min_w \; \sum_i (y_i - \hat y_i)^2 + \alpha \sum_j w_j^2$$

$$\text{Lasso (L1)}:\quad \min_w \; \sum_i (y_i - \hat y_i)^2 + \alpha \sum_j |w_j|$$

| | **L1 – Lasso** | **L2 – Ridge** |
|---|---|---|
| Ceza | $\alpha\sum\lvert w_j\rvert$ | $\alpha\sum w_j^2$ |
| Ağırlıklara etkisi | Bazılarını **tam sıfır** yapar | Hepsini küçültür, sıfır yapmaz |
| Sonuç | **Seyrek** model → otomatik özellik seçimi | Tüm özellikler küçük katkıyla kalır |
| Korelasyonlu özellikler | Birini seçer, diğerlerini atar | Ağırlığı aralarında paylaştırır |
| Geometri | Kısıt bölgesi **elmas** → köşelerde (eksen üstünde) kesişme | Kısıt bölgesi **daire** → köşe yok |

> İkisini birleştiren yönteme **Elastic Net** denir: $\alpha\,[\rho\sum|w_j| + (1-\rho)\sum w_j^2]$.

**Deney:** 50 özellikli bir veri üretiyoruz ama bunlardan yalnızca **5 tanesi** hedefi gerçekten etkiliyor.

In [ ]:
X, y, gercek_katsayilar = make_regression(n_samples=200, n_features=50, n_informative=5,
                                          noise=15, coef=True, random_state=42)
X_egt, X_tst, y_egt, y_tst = train_test_split(X, y, test_size=0.3, random_state=42)

print("Toplam özellik       :", X.shape[1])
print("Etkili özellik sayısı:", int((gercek_katsayilar != 0).sum()))
print("Etkili özellikler    :", np.where(gercek_katsayilar != 0)[0])

In [ ]:
modeller = {
    "LinearRegression": LinearRegression(),
    "Ridge (L2, α=10)": Ridge(alpha=10),
    "Lasso (L1, α=1)": Lasso(alpha=1, max_iter=10_000),
}

satirlar, katsayilar = [], {"Gerçek katsayılar": gercek_katsayilar}
for isim, model in modeller.items():
    model.fit(X_egt, y_egt)
    katsayilar[isim] = model.coef_
    satirlar.append({
        "Model": isim,
        "Eğitim MSE": mean_squared_error(y_egt, model.predict(X_egt)),
        "Test MSE": mean_squared_error(y_tst, model.predict(X_tst)),
        "Tam sıfır katsayı": int(np.sum(np.abs(model.coef_) < 1e-8)),
        "Σ|w|": np.abs(model.coef_).sum(),
    })

pd.DataFrame(satirlar).round(2)

In [ ]:
fig, eksenler = plt.subplots(1, 4, figsize=(18, 3.8), sharey=True)
for ax, (isim, w) in zip(eksenler, katsayilar.items()):
    renkler = ["tab:green" if g != 0 else "tab:gray" for g in gercek_katsayilar]
    ax.bar(range(len(w)), w, color=renkler)
    ax.set_title(isim)
    ax.set_xlabel("özellik indeksi")
eksenler[0].set_ylabel("katsayı değeri")
fig.suptitle("Yeşil = gerçekten etkili özellik, gri = gürültü özelliği", y=1.03)
plt.tight_layout()
plt.show()

### Regularizasyon yolu (regularization path)

$\alpha$ büyüdükçe katsayılar nasıl değişiyor? Lasso'da katsayılar **tek tek sıfıra düşüp orada kalır**, Ridge'de ise hepsi birlikte yavaşça sıfıra **yaklaşır** ama ulaşmaz.

In [ ]:
alfalar = np.logspace(-2, 4, 80)
ridge_yolu = np.array([Ridge(alpha=a).fit(X_egt, y_egt).coef_ for a in alfalar])
lasso_yolu = np.array([Lasso(alpha=a, max_iter=10_000).fit(X_egt, y_egt).coef_ for a in alfalar])

fig, eksenler = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, yol, baslik in zip(eksenler, [ridge_yolu, lasso_yolu], ["Ridge (L2)", "Lasso (L1)"]):
    for j in range(yol.shape[1]):
        etkili = gercek_katsayilar[j] != 0
        ax.plot(alfalar, yol[:, j], color="tab:green" if etkili else "tab:gray",
                lw=2 if etkili else 0.7, alpha=1 if etkili else 0.6)
    ax.set_xscale("log")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_xlabel("α (log ölçek)")
    ax.set_ylabel("katsayı")
    ax.set_title(f"{baslik} regularizasyon yolu")
plt.tight_layout()
plt.show()

sifir_sayisi = (np.abs(lasso_yolu) < 1e-8).sum(axis=1)
plt.figure(figsize=(8, 3.5))
plt.semilogx(alfalar, sifir_sayisi, "o-", ms=3)
plt.axhline(45, color="red", ls=":", label="gürültü özelliği sayısı (45)")
plt.xlabel("α"); plt.ylabel("sıfır katsayı sayısı")
plt.title("Lasso: α arttıkça model seyrekleşir")
plt.legend(); plt.show()

### Lasso ile özellik seçimi

Uygun bir α değerinde Lasso, sıfırdan farklı bıraktığı katsayılarla **hangi özelliklerin önemli olduğunu** bize söyler.

In [ ]:
lasso = Lasso(alpha=1, max_iter=10_000).fit(X_egt, y_egt)
secilen = set(np.where(np.abs(lasso.coef_) > 1e-8)[0])
gercek = set(np.where(gercek_katsayilar != 0)[0])

print("Lasso'nun seçtiği özellikler:", sorted(secilen))
print("Gerçek etkili özellikler    :", sorted(gercek))
print("Doğru yakalanan             :", sorted(secilen & gercek))
print("Yanlışlıkla seçilen         :", sorted(secilen - gercek))

---
## 3) Veri Sızıntısı (Data Leakage) ve `StandardScaler`

**Veri sızıntısı**, modelin eğitim sırasında gerçek hayatta tahmin anında **erişemeyeceği bir bilgiye** erişmesidir. Sonuç: geliştirme ortamında harika görünen skorlar, canlıya çıkınca çöken bir model.

| Sızıntı türü | Örnek |
|---|---|
| **Train–test kirlenmesi** | `StandardScaler`, `SelectKBest`, `SimpleImputer` gibi adımları bölmeden **önce** tüm veriye `fit` etmek |
| **Hedef sızıntısı** (target leakage) | Hedef gerçekleştikten **sonra** oluşan bir sütunu özellik olarak kullanmak |
| **Zamansal sızıntı** | Zaman serisinde geleceğe ait veriyle geçmişi tahmin etmek (rastgele `shuffle` ile bölmek) |
| **Tekrarlanan kayıtlar** | Aynı müşterinin satırları hem eğitimde hem testte |

### `StandardScaler` nasıl çalışır?

$$z = \frac{x - \mu}{\sigma}$$

$\mu$ ve $\sigma$ **sadece eğitim verisinden** hesaplanmalıdır. Test verisi, canlıda gelecek yeni veriyi temsil eder; onun istatistiklerini önceden bilemeyiz.

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_egt, X_tst, y_egt, y_tst = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

# ❌ YANLIŞ: scaler tüm veriye fit ediliyor → test verisinin ortalaması ve std'si eğitime sızıyor
yanlis_scaler = StandardScaler().fit(X)

# ✅ DOĞRU: scaler sadece eğitim verisine fit edilir, teste yalnızca transform uygulanır
scaler = StandardScaler()
X_egt_s = scaler.fit_transform(X_egt)
X_tst_s = scaler.transform(X_tst)

print("Eğitim ortalaması (ilk 3 özellik):", X_egt_s.mean(axis=0)[:3].round(3))
print("Eğitim std'si     (ilk 3 özellik):", X_egt_s.std(axis=0)[:3].round(3))
print("Test ortalaması   (ilk 3 özellik):", X_tst_s.mean(axis=0)[:3].round(3))
print("Test std'si       (ilk 3 özellik):", X_tst_s.std(axis=0)[:3].round(3))
print("\nTest verisinin ortalaması tam 0, std'si tam 1 değil — bu bir hata değil, OLMASI GEREKEN budur.")
print("\nμ farkı (tüm veri vs eğitim) ilk 3 özellik:",
      (yanlis_scaler.mean_ - scaler.mean_)[:3].round(4))

### Dramatik bir sızıntı örneği: özellik seçimi

Tamamen **rastgele** bir veri üretelim: 10.000 gürültü özelliği, rastgele etiketler. Burada öğrenilecek hiçbir şey yok, dürüst bir model ~%50 doğruluk almalı.

Ama özellik seçimini (`SelectKBest`) çapraz doğrulamadan **önce tüm veriye** yaparsak...

In [ ]:
rng = np.random.RandomState(0)
X_gurultu = rng.randn(200, 10_000)
y_gurultu = rng.randint(0, 2, 200)

# ❌ YANLIŞ: özellik seçimi tüm veriyle yapılıyor (test katmanlarındaki etiketler de kullanılıyor)
X_secilmis = SelectKBest(f_classif, k=20).fit_transform(X_gurultu, y_gurultu)
yanlis_skor = cross_val_score(LogisticRegression(max_iter=1000), X_secilmis, y_gurultu, cv=5)

# ✅ DOĞRU: özellik seçimi Pipeline içinde → her katmanda yalnızca o katmanın eğitim kısmına fit edilir
dogru_pipeline = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
dogru_skor = cross_val_score(dogru_pipeline, X_gurultu, y_gurultu, cv=5)

print(f"❌ Sızıntılı doğruluk : {yanlis_skor.mean():.3f}  ← saf gürültüden 'öğrenmiş' gibi görünüyor!")
print(f"✅ Dürüst doğruluk    : {dogru_skor.mean():.3f}  ← yazı-tura seviyesinde, olması gereken bu")

### Hedef sızıntısı (target leakage)

Kredi temerrüdü tahmini yaptığımızı düşünelim. Veride `tahsilat_ekibine_devredildi` diye bir sütun var. Ama bu bilgi **müşteri temerrüde düştükten sonra** oluşuyor. Kredi başvurusu anında bu sütun elimizde olmayacak.

In [ ]:
rng = np.random.RandomState(42)
n = 2000
kredi = pd.DataFrame({
    "gelir": rng.normal(15_000, 5_000, n).round(),
    "borc_orani": rng.uniform(0, 1, n).round(2),
    "yas": rng.randint(20, 65, n),
})
olasilik = 1 / (1 + np.exp(-(3 * kredi["borc_orani"] - kredi["gelir"] / 10_000)))
kredi["temerrut"] = (rng.rand(n) < olasilik).astype(int)
# Temerrüt SONRASI oluşan sütun: temerrüde düşenlerin %95'i tahsilata devredilmiş
kredi["tahsilat_ekibine_devredildi"] = np.where(kredi["temerrut"] == 1, rng.rand(n) < 0.95, rng.rand(n) < 0.02).astype(int)

ozellikler_sizintili = ["gelir", "borc_orani", "yas", "tahsilat_ekibine_devredildi"]
ozellikler_temiz = ["gelir", "borc_orani", "yas"]

for isim, sutunlar in [("Sızıntılı", ozellikler_sizintili), ("Temiz", ozellikler_temiz)]:
    skor = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()),
                           kredi[sutunlar], kredi["temerrut"], cv=5).mean()
    print(f"{isim:10s} özellik seti → CV doğruluğu: {skor:.3f}")

print("\nŞüpheli derecede yüksek bir skor gördüğünüzde ilk soru: 'Bu özellik tahmin anında elimde olacak mı?'")

---
## 4) K-Fold Cross-Validation (Çapraz Doğrulama)

Tek bir train/test bölmesi **şansa bağlıdır**: hangi örneklerin teste düştüğüne göre skor değişir. K-Fold bu belirsizliği azaltır:

1. Veri $k$ eşit parçaya (fold) bölünür.
2. Her turda 1 parça **doğrulama**, kalan $k-1$ parça **eğitim** olur.
3. $k$ turun skorlarının **ortalaması** (ve standart sapması) raporlanır.

```
Fold 1: [TEST ][eğit][eğit][eğit][eğit]
Fold 2: [eğit][TEST ][eğit][eğit][eğit]
Fold 3: [eğit][eğit][TEST ][eğit][eğit]
Fold 4: [eğit][eğit][eğit][TEST ][eğit]
Fold 5: [eğit][eğit][eğit][eğit][TEST ]
```

| Varyant | Ne zaman? |
|---|---|
| `KFold` | Regresyon, dengeli veri |
| `StratifiedKFold` | Sınıflandırma — her fold'da sınıf oranları korunur |
| `GroupKFold` | Aynı kişiye/hastaya ait kayıtlar aynı fold'da kalmalı |
| `TimeSeriesSplit` | Zaman serisi — sadece geçmişle eğitip gelecekle test |

### Tek bölmenin şansa bağlılığı

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

tek_bolme_skorlari = []
for seed in range(100):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=seed)
    tek_bolme_skorlari.append(model.fit(Xa, ya).score(Xb, yb))

cv_skorlari = cross_val_score(model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42))

plt.figure(figsize=(9, 4))
plt.hist(tek_bolme_skorlari, bins=15, alpha=0.7, label="100 farklı rastgele bölme")
plt.axvline(cv_skorlari.mean(), color="red", lw=2, label=f"5-Fold CV ortalaması = {cv_skorlari.mean():.3f}")
plt.xlabel("Test doğruluğu"); plt.ylabel("frekans")
plt.title("Tek train/test bölmesi ne kadar değişken?")
plt.legend(); plt.show()

print(f"Tek bölme skorları: min = {min(tek_bolme_skorlari):.3f}, max = {max(tek_bolme_skorlari):.3f}")
print(f"5-Fold CV        : {cv_skorlari.mean():.3f} ± {cv_skorlari.std():.3f}")

### K-Fold döngüsünü elle yazmak

`cross_val_score` arka planda tam olarak bunu yapar. Dikkat: scaler **her fold içinde yeniden** fit ediliyor.

In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_skorlari = []

for fold, (egitim_idx, dogrulama_idx) in enumerate(kf.split(X, y), start=1):
    X_f_egt, X_f_dgr = X[egitim_idx], X[dogrulama_idx]
    y_f_egt, y_f_dgr = y[egitim_idx], y[dogrulama_idx]

    scaler = StandardScaler().fit(X_f_egt)            # sadece bu fold'un eğitim kısmı
    clf = LogisticRegression(max_iter=1000).fit(scaler.transform(X_f_egt), y_f_egt)
    skor = accuracy_score(y_f_dgr, clf.predict(scaler.transform(X_f_dgr)))
    fold_skorlari.append(skor)

    print(f"Fold {fold}: eğitim = {len(egitim_idx)}, doğrulama = {len(dogrulama_idx)}, "
          f"pozitif oranı = {y_f_dgr.mean():.2f}, doğruluk = {skor:.3f}")

print(f"\nOrtalama: {np.mean(fold_skorlari):.3f} ± {np.std(fold_skorlari):.3f}")

### `KFold` vs `StratifiedKFold` — dengesiz veride fark

%10 pozitif sınıf içeren bir veride düz `KFold` bazı fold'lara çok az pozitif örnek düşürebilir.

In [ ]:
X_d, y_d = make_classification(n_samples=200, weights=[0.9, 0.1], random_state=1)
siralama = np.argsort(y_d)                    # etiketleri sıralı hale getir (kötü senaryo)
X_d, y_d = X_d[siralama], y_d[siralama]

fig, eksenler = plt.subplots(1, 2, figsize=(15, 3))
for ax, (isim, bolucu) in zip(eksenler, [("KFold (shuffle yok)", KFold(5)),
                                          ("StratifiedKFold", StratifiedKFold(5))]):
    matris = np.zeros((5, len(y_d)))
    oranlar = []
    for i, (_, dgr) in enumerate(bolucu.split(X_d, y_d)):
        matris[i, dgr] = 1
        oranlar.append(y_d[dgr].mean())
    ax.imshow(matris, aspect="auto", cmap="Blues", interpolation="nearest")
    ax.set_yticks(range(5))
    ax.set_yticklabels([f"Fold {i+1} (pozitif %{o*100:.0f})" for i, o in enumerate(oranlar)])
    ax.set_xlabel("örnek indeksi (koyu = doğrulama)")
    ax.set_title(isim)
plt.tight_layout()
plt.show()

### Hepsi bir arada: Pipeline + GridSearchCV

Regularizasyon gücünü (`C = 1/α`) seçmek için çapraz doğrulama kullanıyoruz. `Pipeline` sayesinde scaler her fold'da doğru şekilde fit ediliyor, yani **sızıntı yok**. Son değerlendirme ise GridSearch'ün hiç görmediği test setinde yapılıyor.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, solver="liblinear")),
])
param_izgara = {
    "model__C": np.logspace(-3, 2, 11),
    "model__penalty": ["l1", "l2"],
}
arama = GridSearchCV(pipeline, param_izgara, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                     scoring="accuracy", n_jobs=-1)
arama.fit(X_egt, y_egt)

print("En iyi parametreler:", arama.best_params_)
print(f"En iyi CV skoru     : {arama.best_score_:.3f}")
print(f"Test skoru          : {arama.score(X_tst, y_tst):.3f}")

sonuclar = pd.DataFrame(arama.cv_results_)
plt.figure(figsize=(9, 4))
for ceza in ["l1", "l2"]:
    alt = sonuclar[sonuclar["param_model__penalty"] == ceza]
    plt.semilogx(alt["param_model__C"].astype(float), alt["mean_test_score"], "o-", label=ceza.upper())
plt.xlabel("C (küçük C = güçlü regularizasyon)")
plt.ylabel("ortalama CV doğruluğu")
plt.title("Regularizasyon gücü vs CV performansı")
plt.legend(); plt.show()

---
## Özet

| Kavram | Akılda kalacak kural |
|---|---|
| Underfitting | Eğitim hatası da yüksek → modeli güçlendir |
| Overfitting | Eğitim–test arasında büyük boşluk → regularizasyon, daha fazla veri, daha basit model |
| L1 / Lasso | Seyrek model, otomatik özellik seçimi |
| L2 / Ridge | Tüm ağırlıkları küçültür, korelasyonlu özelliklerde kararlı |
| Veri sızıntısı | Her ön işleme adımı **sadece eğitim verisine** `fit` edilir → `Pipeline` kullan |
| `StandardScaler` | `fit_transform(X_train)`, `transform(X_test)` — asla `fit(X_tum)` |
| K-Fold CV | Tek bölme yerine k bölmenin ortalaması ± std; sınıflandırmada `StratifiedKFold` |